# Trainable bubble segmentation (hand annotation → Mask R-CNN)

This pipeline learns from bubbles **you** draw. It is built for the three hard parts of this data:

| problem | how it is handled |
|---|---|
| **overlapping / nested bubbles** | Mask R-CNN predicts a separate mask per bubble, so masks may overlap (Cellpose cannot do this). Annotations are napari *Shapes*, one per bubble, also allowed to overlap. |
| **non-spherical bubbles** | masks are free-form; you annotate with ellipses (rotatable) or polygons. Output includes area, equivalent radius and fitted-ellipse axes. |
| **large size range (≈2–90 px diameter)** | frames are upsampled ×3 (`model_scale`); the feature pyramid has anchors from 12 to 192 px; training uses scale-jitter augmentation (×0.6–1.6). |

Model input = 3 channels: normalised image, CLAHE, and the rim (dark-ridge) map from `bubble_seg.py`.

**Workflow (repeat 2–3 times):**
1. **Stage** a few frames (section 2).
2. **Annotate** them in napari (section 3, a Maxwell desktop session). Draw a yellow "fully annotated" rectangle around each region where you drew *every* bubble.
3. **Train** (sections 4–5). This starts from COCO-pretrained weights; ~3000 iterations take ~10–20 min on an A100.
4. **Evaluate** on held-out regions (section 6) and **predict** new frames (section 7).
5. **Correct** predictions on new frames instead of drawing from scratch (section 8), then retrain. Each round is faster
   than the last.

**How much to annotate first:** ~150–300 bubbles in 3–5 rectangles spread over 2–3 frames (early / mid / late
in a train). Include overlapping clusters, the smallest bubbles, the largest ones, and the fibre region.
Precision matters more than quantity.

## 0. Setup (DESY Maxwell)

In the same venv as the other notebook (`~/venvs/foam`):
```bash
pip install -r requirements.txt          # includes torchvision + tifffile
pip install "napari[all]"                # annotation GUI (only needed where you annotate)
# download the COCO-pretrained Mask R-CNN weights once (~180 MB, needs internet, e.g. a login node):
python -c "from torchvision.models.detection import maskrcnn_resnet50_fpn_v2 as m; m(weights='DEFAULT')"
```
Run this notebook on a **GPU** JupyterHub session. Annotation needs a desktop (napari is a Qt window); see section 3.

In [ ]:
import os, glob, time, dataclasses
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2
import bubble_seg as bs
import bubble_rcnn as br

import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
plt.rcParams["figure.dpi"] = 90

## CONFIG

Lengths are in **model pixels** (= native × `model_scale`) unless noted. Things you are most likely to change:
`iterations`, `batch_size`, `crop_size`, `score_thresh`, `mask_nms_iou`, `model_scale` and `anchor_sizes`.

In [ ]:
FRAME_GLOB = "raw/jpg_r563_svd_normalised_tr050/*.jpg"   # clean native frames (or the tiff_ folder)

cfg = br.RCNNConfig(
    annotation_dir="annotations",          # staged frames + your JSON annotations
    model_dir="models",
    model_scale=3.0,                       # native -> model px; smallest bubbles (~2 px) become ~6 px
    channels=("norm", "clahe", "ridge"),   # model input channels
    val_fraction=0.2,                      # share of annotated rectangles held out for validation

    # model
    pretrained=True,
    anchor_sizes=((12,), (24,), (48,), (96,), (192,)),   # model px, 5 pyramid levels
    anchor_ratios=(0.5, 1.0, 2.0),
    box_nms_thresh=0.7,                    # keep high, overlapping bubbles have overlapping boxes
    detections_per_img=400,

    # training
    device="cuda",
    crop_size=384, batch_size=4, iterations=3000,
    lr=0.01, warmup_iters=200, weight_decay=1e-4, amp=True,
    num_workers=4, val_every=250,
    scale_jitter=(0.6, 1.6), rotate90=True, flip=True,
    intensity_jitter=0.15, noise_std=0.03, min_instance_area=12.0,

    # inference
    score_thresh=0.5,                      # lower -> more bubbles (and more false ones)
    mask_thresh=0.5,
    mask_nms_iou=0.7,                      # duplicates have mask IoU > ~0.8; real overlaps are usually < 0.6
    um_per_px=3.2,

    # bubbles cut by the image edge: fitted from their visible arc, centre may lie outside the image
    edge_min_arc_deg=100.0,                # shorter visible arcs always get a circle
    edge_ellipse_gain=1.6,                 # ellipse only if it fits the arc this much better than a circle
    edge_reliable_arc_deg=140.0,           # shorter arcs -> fit_reliable=False
    # optional exclusions (rows stay in the table, in_analysis=False)
    drop_unreliable_fits=False,            # exclude edge bubbles with fit_reliable=False
    drop_edge_bubbles=False,               # exclude every bubble touching the image edge
    inner_margin=0.0,                      # native px; exclude bubbles whose centre is closer than this to the edge
)
os.makedirs(cfg.annotation_dir, exist_ok=True)
os.makedirs(cfg.model_dir, exist_ok=True)

## 1. Look at the model input

Check that the three channels look sensible. The rim map should show the bubble outlines clearly.

In [ ]:
frames = sorted(glob.glob(FRAME_GLOB))
print(f"{len(frames)} frames found")
img0 = bs.load_image(frames[0])
x0 = br.prepare_image(img0, cfg)
fig, ax = plt.subplots(1, 4, figsize=(20, 4))
ax[0].imshow(img0, cmap="gray"); ax[0].set_title(f"native {img0.shape}")
for a, ch, name in zip(ax[1:], x0, cfg.channels):
    a.imshow(ch, cmap="gray" if name != "ridge" else "magma"); a.set_title(f"{name} (model res {ch.shape})")
for a in ax: a.axis("off")
plt.tight_layout()

## 2. Stage frames for annotation

Copies the chosen frames into `annotations/` (float32 TIFF) and creates an empty JSON per frame.
Re-running is safe: existing annotations are never overwritten. Spread the picks over time, because bubbles grow and shrink.

In [ ]:
import re

# ---- initial frames to annotate: (train, frame) pairs, or strings like "train0frame010" / "tid0_010" ----
INITIAL_FRAMES = [
    (0, 10),
    (3, 36),
    (7, 214),
]
PREVIEW = True     # show the frames first (check exposure)
STAGE = False       # create .tif + empty .json in annotations/ (existing files are never overwritten)

# helpers (also used by section 8)
_pat = re.compile(r"_tid(\d+)_(\d+)\.[^.]+$")
frame_index = {}
for f in frames:
    m = _pat.search(os.path.basename(f))
    if m:
        frame_index[(int(m.group(1)), int(m.group(2)))] = f

def parse_sel(s):
    if isinstance(s, (tuple, list)):
        return int(s[0]), int(s[1])
    m = re.fullmatch(r"\s*(?:train|tid)?\s*(\d+)\s*[_,\s]*(?:frame|f)?\s*(\d+)\s*", str(s), re.I)
    if not m:
        raise ValueError(f"cannot read frame selection {s!r} (use (train, frame) or 'train0frame010')")
    return int(m.group(1)), int(m.group(2))

def select_frames(selection):
    out = []
    for s in selection:
        key = parse_sel(s)
        if key in frame_index:
            out.append(frame_index[key])
        else:
            print(f"not found: train {key[0]} frame {key[1]}  (check FRAME_GLOB / numbering)")
    print("selected:", [os.path.basename(f) for f in out])
    return out

def preview_frames(paths):
    if not paths:
        return
    fig, axs = plt.subplots(1, len(paths), figsize=(5 * len(paths), 3.5), squeeze=False)
    for ax, f in zip(axs[0], paths):
        im = bs.load_image(f)
        ax.imshow(im, cmap="gray", vmin=0, vmax=1)    # fixed range: over/under-exposure stays visible
        ax.set_title(f"{os.path.basename(f)}\nmean {im.mean():.2f}, clipped {100 * np.mean((im < 0.01) | (im > 0.99)):.1f}%", fontsize=8)
        ax.axis("off")
    plt.tight_layout(); plt.show()

init_frames = select_frames(INITIAL_FRAMES)
if PREVIEW:
    preview_frames(init_frames)
if STAGE:
    staged = br.stage_frames_for_annotation(init_frames, cfg)
    print("\nAnnotate in a FastX terminal, from the repo folder:  python annotate_bubbles.py")
    for js in staged:
        print(f"  or directly: python {os.path.abspath('annotate_bubbles.py')} {os.path.abspath(js)}")

## 3. Annotate in napari (outside the notebook)

**Where:** napari needs a desktop. On Maxwell use a **FastX desktop session** (https://max-display.desy.de:3443 →
start a desktop → open a terminal). Alternatively copy `annotations/` to your laptop, annotate there (same venv
packages), and copy the `.json` files back.
```bash
source ~/venvs/foam/bin/activate
cd <this repo>
python annotate_bubbles.py annotations/<frame>.json
# if the window is black / OpenGL errors in FastX:  LIBGL_ALWAYS_SOFTWARE=1 python annotate_bubbles.py ...
```

**How:**
* Select the **bubbles** layer. Press **E** for an ellipse or **P** for a polygon (double-click to finish). Draw one shape per bubble.
  Rotate or resize an ellipse with the select tool (**S**). Change the line width in the layer controls if needed.
* Draw the **full outline** of every bubble, *including parts hidden behind other bubbles or fibres*: continue the
  visible arc. Overlapping shapes are expected. A bubble seen inside a bigger one gets its own shape, and the big one
  keeps its full outline too.
* Toggle the **rim map** layer (eye icon) to see faint rims.
* Select the **fully annotated** layer and press **R** to draw a rectangle around each region where you have drawn
  *every* bubble. Only these regions are used for training, so bubbles you skipped elsewhere do no harm.
  No rectangle means the whole frame counts as complete.
* Save with the **Save** button or **Shift-S**. It also autosaves every 2 min and on close.

## 4. Load and check the annotations

In [ ]:
anns = br.list_annotations(cfg)
assert anns, f"no annotations with bubbles in {cfg.annotation_dir}/ yet (see sections 2-3)"
print(f"{len(anns)} annotated frames, {sum(len(a['bubbles']) for a in anns)} bubbles, "
      f"{sum(max(1, len(a['rois'])) for a in anns)} training regions")
for a in anns:
    br.plot_annotation(a)

In [ ]:
samples = br.build_samples(anns, cfg)
train_samples, val_samples = br.split_samples(samples, cfg)
print("train:", [f"{s.name} ({len(s.masks)})" for s in train_samples])
print("val:  ", [f"{s.name} ({len(s.masks)})" for s in val_samples])

r_eq = np.concatenate([np.sqrt(s.masks.reshape(len(s.masks), -1).sum(1) / np.pi) / cfg.model_scale for s in samples if len(s.masks)])
plt.figure(figsize=(6, 3)); plt.hist(r_eq * cfg.um_per_px, bins=40)
plt.xlabel("annotated equivalent radius [µm]"); plt.ylabel("count"); plt.tight_layout()

Augmentation preview (what the network sees during training):

In [ ]:
rng = np.random.default_rng(0)
fig, axs = plt.subplots(2, 4, figsize=(18, 9))
for ax in axs.ravel():
    im, m = br.augment(train_samples[rng.integers(len(train_samples))], cfg, rng)
    ax.imshow(im[0], cmap="gray")
    for mm in m:
        for c in br.measure.find_contours(np.pad(mm, 1).astype(float), 0.5):
            ax.plot(c[:, 1] - 1, c[:, 0] - 1, lw=0.7)
    ax.set_title(f"{len(m)} instances"); ax.axis("off")
plt.tight_layout()

## 5. Train

The best checkpoint (by validation F1) is saved as `models/bubble_rcnn_best.pt`, the final one as `_last.pt`.
With very few annotations, validation numbers jump around. Judge mostly by the overlays in sections 6–7.

In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    model = br.build_model(cfg)
    history = br.train(model, train_samples, val_samples, cfg)
    history.to_csv(os.path.join(cfg.model_dir, "train_history.csv"), index=False)

    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    ax[0].plot(history["iter"], history["loss"].rolling(25, min_periods=1).mean())
    ax[0].set_xlabel("iteration"); ax[0].set_ylabel("loss (smoothed)"); ax[0].set_yscale("log")
    if "val_f1" in history:
        v = history.dropna(subset=["val_f1"])
        for k in ["val_precision", "val_recall", "val_f1"]:
            ax[1].plot(v["iter"], v[k], "o-", label=k)
        ax[1].set_ylim(0, 1); ax[1].legend(); ax[1].set_xlabel("iteration")
    plt.tight_layout()

## 6. Evaluate on the held-out regions

In [ ]:
MODEL_PATH = os.path.join(cfg.model_dir, "bubble_rcnn_best.pt" if val_samples else "bubble_rcnn_last.pt")
model, mcfg = br.load_model(MODEL_PATH)
# inference / measurement settings come from CONFIG (the model file only fixes the network + preprocessing)
INFER_KEYS = ["score_thresh", "mask_nms_iou", "mask_thresh", "um_per_px", "edge_min_arc_deg", "edge_ellipse_gain",
              "edge_max_axis_ratio", "edge_reliable_arc_deg", "drop_unreliable_fits", "drop_edge_bubbles", "inner_margin"]
mcfg = dataclasses.replace(mcfg, **{k: getattr(cfg, k) for k in INFER_KEYS})

if val_samples:
    ev = br.evaluate(model, val_samples, mcfg, return_details=True)
    print({k: round(v, 3) if isinstance(v, float) else v for k, v in ev.items() if k != "matches"})
    for s in val_samples:
        pred = br.predict(model, s.image, mcfg)
        fig, ax = plt.subplots(1, 2, figsize=(16, 7))
        for a, (masks, t) in zip(ax, [(s.masks, f"annotation ({len(s.masks)})"), (pred["masks"], f"prediction ({len(pred['masks'])})")]):
            a.imshow(s.image[0], cmap="gray")
            for i, mm in enumerate(masks):
                for c in br.measure.find_contours(np.pad(mm, 1).astype(float), 0.5):
                    a.plot(c[:, 1] - 1, c[:, 0] - 1, lw=0.8, color=plt.cm.tab20(i % 20))
            a.set_title(f"{s.name}: {t}"); a.axis("off")
        plt.tight_layout()
    m = ev["matches"]
    if len(m):
        plt.figure(figsize=(5, 5))
        plt.scatter(m["r_gt"] * mcfg.um_per_px, m["r_pred"] * mcfg.um_per_px, s=10)
        lim = [0, m[["r_gt", "r_pred"]].max().max() * mcfg.um_per_px * 1.05]
        plt.plot(lim, lim, "k--", lw=0.8); plt.xlabel("annotated r_eq [µm]"); plt.ylabel("predicted r_eq [µm]")
        plt.title(f"matched bubbles, median IoU {m['iou'].median():.2f}"); plt.tight_layout()

## 7. Predict a new frame

Tune `score_thresh` (CONFIG, then re-run from section 6) by eye. Lower it if bubbles are missing, raise it if there are
false detections.

**Bubbles cut by the image edge** are measured from an ellipse or circle fitted to their *visible arc*, so `x, y`
(which may lie outside the image), `area_px2`, `r_eq` and the axes describe the **full** bubble.
* `edge_truncated`: the bubble touches the image edge.
* `fit_kind`: `mask` (interior bubble, measured from its mask), `circle` or `ellipse`.
* `arc_deg`: how much of the outline is visible.
* `fit_reliable`: False when the visible arc is short (< `edge_reliable_arc_deg`).
* `area_visible_px2`: the part inside the image.
* `in_analysis`: False for bubbles excluded by `drop_unreliable_fits`, `drop_edge_bubbles` or `inner_margin` (internal boundary).

Use `df[df["in_analysis"]]` for statistics.

In [ ]:
PREDICT_FRAME = frames[len(frames) // 2 - 150]
t = time.time()
img, x, pred, df = br.predict_frame(model, PREDICT_FRAME, mcfg)
print(f"{len(df)} bubbles in {time.time() - t:.1f}s  ({os.path.basename(PREDICT_FRAME)})")
n_edge = int(df["edge_truncated"].sum()) if len(df) else 0
n_in = int(df["in_analysis"].sum()) if len(df) else 0
print(f"{n_edge} cut by the image edge (fitted from their visible arc), {n_in} in analysis")
VIEW_PAD = 40   # native px shown beyond the image so the fitted outlines of edge bubbles are visible
br.plot_instances(img, pred, mcfg, df=df, view_pad=VIEW_PAD,
                  title=f"{os.path.basename(PREDICT_FRAME)}: {len(df)} bubbles (dashed: fitted edge outlines, grey: excluded)");

In [ ]:
display(df.round(3).head(15))
dfa = df[df["in_analysis"]] if len(df) else df     # bubbles used for statistics (exclusions applied)
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
if len(dfa):
    ax[0].hist(dfa["r_eq_um"], bins=40); ax[0].set_xlabel("equivalent radius [µm]"); ax[0].set_ylabel("count")
    ax[1].scatter(dfa["r_eq_um"], dfa["minor_axis"] / dfa["major_axis"].clip(lower=1e-6), s=10, c=dfa["score"], cmap="viridis")
    ax[1].set_xlabel("equivalent radius [µm]"); ax[1].set_ylabel("minor / major axis"); ax[1].set_title("colour: score")
plt.tight_layout()
os.makedirs("outputs", exist_ok=True)
df.to_csv(os.path.join("outputs", os.path.splitext(os.path.basename(PREDICT_FRAME))[0] + "_rcnn_bubbles.csv"), index=False)  # all rows

## 8. Correct predictions → more training data

Stage new frames with the model's predictions pre-drawn. In napari you then only fix mistakes: delete false shapes,
add missed bubbles, adjust outlines. Then draw the yellow "fully annotated" rectangles. Frames that already have
annotations are skipped, so nothing is overwritten. After correcting, re-run from section 4.

`kind="ellipse"` gives shapes that are quick to adjust; `kind="polygon"` keeps the exact predicted outline.

In [ ]:
# ---- more frames for the next training round (same formats as in section 2) ----
SELECTED = [
    (1, 50),
    (5, 120),
]
PREVIEW = True              # show the frames first (check exposure)
MAKE_DRAFTS = False          # stage .tif + .json (set False to only preview)
WITH_PREDICTIONS = False     # True: pre-fill with model predictions to correct; False: empty JSON to draw from scratch
DRAFT_SCORE = 0.4           # score threshold for drafted shapes (a bit below score_thresh)
DRAFT_KIND = "ellipse"      # "ellipse" (easy to adjust) or "polygon" (exact predicted outline)

DRAFT_FRAMES = select_frames(SELECTED)      # helpers defined in section 2
if PREVIEW:
    preview_frames(DRAFT_FRAMES)

if MAKE_DRAFTS:
    dcfg = dataclasses.replace(mcfg, score_thresh=DRAFT_SCORE)
    for f in DRAFT_FRAMES:
        js = br.stage_frames_for_annotation([f], cfg)[0]   # creates the .tif + empty .json only if missing
        if len(br.load_annotation(js)["bubbles"]):
            print("already annotated, left untouched:", os.path.basename(js))
            continue
        if WITH_PREDICTIONS:
            _, _, p, _ = br.predict_frame(model, f, dcfg)
            shapes = br.masks_to_shapes(p, dcfg, kind=DRAFT_KIND)
            br.write_prediction_annotation(js, os.path.basename(js).replace(".json", ".tif"), shapes)
            print(f"draft with {len(shapes):3d} predicted bubbles: {os.path.abspath(js)}")
        else:
            print(f"empty annotation ready: {os.path.abspath(js)}")
    print("\nopen them one by one with:  python annotate_bubbles.py   (lists frames, Enter = first unannotated)")